# 06. Transformer (rubert-base) для определения границ предложений

**Цель:** превзойти CatBoost (F1=0.984) с помощью более мощного трансформера.

**Модель:** `DeepPavlov/rubert-base-cased` — 178M параметров, контекст 512 токенов.

**Отличия от notebook 05 (rubert-tiny2, F1≈0.953):**
- Модель в 6× больше (178M vs 29M) — лучше понимает контекст диалогов, кавычек, тире
- Контекст 512 токенов → sliding window для длинных текстов
- Усиленный вес границ (scale=1.0) и увеличенный patience

In [1]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
import json
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 12

SEED = 42
MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LEN = 512
STRIDE = 256  # overlap between sliding window chunks
DATA_PATH = Path("../data/processed/sentences.jsonl")
EXCLUDE_IDS = {"text_833", "text_843"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

with open(DATA_PATH, encoding="utf-8") as f:
    dataset = [json.loads(line) for line in f]
dataset = [r for r in dataset if r["id"] not in EXCLUDE_IDS]
print(f"Текстов: {len(dataset)}")
print(f"Предложений: {sum(r['num_sentences'] for r in dataset)}")

Device: mps
Текстов: 532
Предложений: 20740


## 1. Train/test split

Точно такой же split, как в notebook 04 (CatBoost) и 05 (rubert-tiny2), для честного сравнения.

In [2]:
random.seed(SEED)
text_ids = [r["id"] for r in dataset]
random.shuffle(text_ids)

split_idx = int(len(text_ids) * 0.7)
train_ids = set(text_ids[:split_idx])
test_ids = set(text_ids[split_idx:])

train_data = [r for r in dataset if r["id"] in train_ids]
test_data = [r for r in dataset if r["id"] in test_ids]

# fit/val split (80/20 от train, как в notebook 04)
random.seed(SEED)
train_text_ids_list = [r["id"] for r in train_data]
random.shuffle(train_text_ids_list)
val_split = int(len(train_text_ids_list) * 0.8)
fit_ids = set(train_text_ids_list[:val_split])
val_ids = set(train_text_ids_list[val_split:])

fit_data = [r for r in train_data if r["id"] in fit_ids]
val_data = [r for r in train_data if r["id"] in val_ids]

print(f"Train: {len(train_data)} текстов, {sum(r['num_sentences'] for r in train_data)} предложений")
print(f"  Fit: {len(fit_data)}, Val: {len(val_data)}")
print(f"Test:  {len(test_data)} текстов, {sum(r['num_sentences'] for r in test_data)} предложений")

Train: 372 текстов, 14525 предложений
  Fit: 297, Val: 75
Test:  160 текстов, 6215 предложений


## 2. Токенизация и sliding window

`rubert-base-cased` имеет контекст 512 токенов. Тексты длиннее разбиваем на перекрывающиеся окна (stride=256).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token_counts = []
for rec in dataset:
    n_tokens = len(tokenizer(rec["clean_text"], add_special_tokens=True)["input_ids"])
    token_counts.append(n_tokens)

token_counts = np.array(token_counts)
content_len = MAX_LEN - 2  # [CLS] + [SEP]
n_over = (token_counts > MAX_LEN).sum()

print(f"Токенов: min={token_counts.min()}, median={np.median(token_counts):.0f}, "
      f"P95={np.percentile(token_counts, 95):.0f}, max={token_counts.max()}")
print(f"Текстов > {MAX_LEN} токенов: {n_over} ({100*n_over/len(token_counts):.0f}%) → нужен sliding window")

# Демо
sample = dataset[0]
enc = tokenizer(sample["clean_text"], return_offsets_mapping=True, add_special_tokens=False)
if len(enc["input_ids"]) > content_len:
    starts = list(range(0, len(enc["input_ids"]) - content_len, STRIDE))
    if starts[-1] + content_len < len(enc["input_ids"]):
        starts.append(len(enc["input_ids"]) - content_len)
    n_chunks = len(starts)
else:
    n_chunks = 1
print(f"\nПример: {len(sample['clean_text'])} символов → {len(enc['input_ids'])} токенов → {n_chunks} chunk(s)")

## 3. Dataset и DataLoader (sliding window)

Длинные тексты разбиваются на перекрывающиеся окна по 512 токенов (stride=256).
Каждое окно — отдельный training example. Границы маппятся через `offset_mapping`.

In [ ]:
def get_true_boundaries(rec):
    sents = rec["sentences"]
    return set(sents[i]["end"] for i in range(len(sents) - 1))


class SentenceBoundaryDataset(Dataset):
    def __init__(self, records, tokenizer, max_len=MAX_LEN, stride=STRIDE):
        self.items = []
        cls_id = tokenizer.cls_token_id
        sep_id = tokenizer.sep_token_id
        content_len = max_len - 2

        for rec in records:
            enc = tokenizer(
                rec["clean_text"],
                return_offsets_mapping=True,
                add_special_tokens=False,
            )
            boundary_positions = get_true_boundaries(rec)
            tokens = enc["input_ids"]
            offsets = enc["offset_mapping"]

            # Compute chunk start positions
            if len(tokens) <= content_len:
                starts = [0]
            else:
                starts = list(range(0, len(tokens) - content_len, stride))
                last_start = len(tokens) - content_len
                if not starts or starts[-1] != last_start:
                    starts.append(last_start)

            for start in starts:
                end = min(start + content_len, len(tokens))
                chunk_offsets = offsets[start:end]

                input_ids = [cls_id] + tokens[start:end] + [sep_id]
                attention_mask = [1] * len(input_ids)

                # Labels: -100 for [CLS]/[SEP], 0 for content tokens
                labels = [-100]
                for cs, ce in chunk_offsets:
                    labels.append(0)
                labels.append(-100)

                # Map boundaries to token labels
                chunk_char_start = chunk_offsets[0][0] if chunk_offsets else 0
                chunk_char_end = chunk_offsets[-1][1] if chunk_offsets else 0
                relevant_bps = {bp for bp in boundary_positions
                               if chunk_char_start <= bp <= chunk_char_end + 5}

                for bp in relevant_bps:
                    found = False
                    for i, (cs, ce) in enumerate(chunk_offsets):
                        if cs <= bp < ce:
                            labels[i + 1] = 1  # +1 for [CLS]
                            found = True
                            break
                    if not found:
                        # Boundary on space — find next token
                        for i, (cs, ce) in enumerate(chunk_offsets):
                            if cs > bp and cs != ce:
                                labels[i + 1] = 1
                                break

                self.items.append({
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                    "labels": labels,
                })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


def collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    pad_id = tokenizer.pad_token_id
    input_ids, attention_mask, labels = [], [], []
    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [pad_id] * pad_len)
        attention_mask.append(item["attention_mask"] + [0] * pad_len)
        labels.append(item["labels"] + [-100] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


fit_dataset = SentenceBoundaryDataset(fit_data, tokenizer)
val_dataset = SentenceBoundaryDataset(val_data, tokenizer)

all_labels = [l for item in fit_dataset.items for l in item["labels"] if l != -100]
pos_count = sum(1 for l in all_labels if l == 1)
neg_count = sum(1 for l in all_labels if l == 0)
pos_ratio = pos_count / len(all_labels)
print(f"Chunks: fit={len(fit_dataset)}, val={len(val_dataset)}")
print(f"Токен-level баланс: {pos_count} positive ({100*pos_ratio:.1f}%), {neg_count} negative")
print(f"Соотношение neg/pos: {neg_count/pos_count:.1f}")

Chunks: fit=672, val=178
Токен-level баланс: 18294 positive (5.4%), 319929 negative
Соотношение neg/pos: 17.5


## 4. Обучение

Fine-tune `rubert-base-cased` с weighted CrossEntropyLoss.
Early stopping по F1 на валидации (patience=7).

In [ ]:
# --- Eval functions + predict with sliding window ---

def evaluate(records, predict_fn, tolerance=2):
    total_tp = total_fp = total_fn = 0
    for rec in records:
        true_bd = get_true_boundaries(rec)
        pred_bd = predict_fn(rec["clean_text"])

        matched_true = set()
        matched_pred = set()
        for pb in pred_bd:
            for tb in true_bd:
                if abs(pb - tb) <= tolerance and tb not in matched_true:
                    matched_true.add(tb)
                    matched_pred.add(pb)
                    break

        total_tp += len(matched_pred)
        total_fp += len(pred_bd) - len(matched_pred)
        total_fn += len(true_bd) - len(matched_true)

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {"TP": total_tp, "FP": total_fp, "FN": total_fn,
            "Precision": precision, "Recall": recall, "F1": f1}


def predict_boundaries(text, model, tokenizer, threshold=0.5):
    model_device = next(model.parameters()).device
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    tokens = enc["input_ids"]
    offsets = enc["offset_mapping"]
    content_len = MAX_LEN - 2
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id

    if len(tokens) <= content_len:
        starts = [0]
    else:
        starts = list(range(0, len(tokens) - content_len, STRIDE))
        last_start = len(tokens) - content_len
        if not starts or starts[-1] != last_start:
            starts.append(last_start)

    # Collect probabilities per token (average over overlapping windows)
    token_probs = {}
    for start in starts:
        end = min(start + content_len, len(tokens))
        chunk_ids = [cls_id] + tokens[start:end] + [sep_id]
        chunk_mask = [1] * len(chunk_ids)

        input_ids_t = torch.tensor([chunk_ids], device=model_device)
        attention_mask_t = torch.tensor([chunk_mask], device=model_device)

        with torch.no_grad():
            logits = model(input_ids=input_ids_t, attention_mask=attention_mask_t).logits[0]
        probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

        for i in range(end - start):
            global_idx = start + i
            prob = float(probs[i + 1])  # +1 to skip [CLS]
            if global_idx not in token_probs:
                token_probs[global_idx] = []
            token_probs[global_idx].append(prob)

    boundaries = set()
    for idx, prob_list in token_probs.items():
        avg_prob = sum(prob_list) / len(prob_list)
        if avg_prob >= threshold:
            cs, ce = offsets[idx]
            if cs != ce:
                boundaries.add(cs)
    return boundaries


def make_predict_fn(model, tokenizer, threshold):
    def predict_fn(text):
        return predict_boundaries(text, model, tokenizer, threshold)
    return predict_fn

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.gradient_checkpointing_enable()  # recompute activations to save MPS memory
model.to(device)

# Weighted loss: полный вес дисбаланса
scale = 1.0
weight = torch.tensor([1.0, (neg_count / pos_count) * scale], device=device)
criterion = nn.CrossEntropyLoss(weight=weight, ignore_index=-100)
print(f"Loss weights: [1.0, {weight[1].item():.1f}] (scale={scale})")

EPOCHS = 30
BATCH_SIZE = 1
GRAD_ACCUM = 16  # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LR = 3e-5
WARMUP_RATIO = 0.1

fit_loader = DataLoader(fit_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(fit_loader) * EPOCHS // GRAD_ACCUM
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
print(f"Epochs: {EPOCHS}, batch_size: {BATCH_SIZE}, grad_accum: {GRAD_ACCUM} (effective: {BATCH_SIZE*GRAD_ACCUM})")
print(f"Optimizer steps: {total_steps}, warmup: {warmup_steps}")
print(f"Fit chunks: {len(fit_dataset)}, batches/epoch: {len(fit_loader)}")

# --- Training loop ---
best_val_f1 = 0
best_epoch = 0
patience = 7
patience_counter = 0
history = {"train_loss": [], "val_f1": [], "val_precision": [], "val_recall": []}
best_state = None

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    n_batches = 0
    optimizer.zero_grad()

    for step, batch in enumerate(fit_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits.view(-1, 2), labels.view(-1))
        loss = loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(fit_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            if device.type == "mps":
                torch.mps.empty_cache()

        total_loss += loss.item() * GRAD_ACCUM
        n_batches += 1

    avg_loss = total_loss / n_batches

    # Validation
    model.eval()
    pred_fn = make_predict_fn(model, tokenizer, 0.5)
    val_result = evaluate(val_data, pred_fn)

    history["train_loss"].append(avg_loss)
    history["val_f1"].append(val_result["F1"])
    history["val_precision"].append(val_result["Precision"])
    history["val_recall"].append(val_result["Recall"])

    print(f"Epoch {epoch+1:2d}/{EPOCHS}: loss={avg_loss:.4f}  "
          f"val P={val_result['Precision']:.3f} R={val_result['Recall']:.3f} F1={val_result['F1']:.3f}  "
          f"FP={val_result['FP']} FN={val_result['FN']}")

    if val_result["F1"] > best_val_f1:
        best_val_f1 = val_result["F1"]
        best_epoch = epoch + 1
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}. Best: epoch {best_epoch} (F1={best_val_f1:.3f})")
            break

print(f"\nBest epoch: {best_epoch}, val F1: {best_val_f1:.3f}")

## 5. Кривые обучения

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], "o-", ms=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Train Loss")

ax2.plot(epochs_range, history["val_f1"], "o-", label="F1", ms=4)
ax2.plot(epochs_range, history["val_precision"], "s--", alpha=0.7, label="Precision", ms=3)
ax2.plot(epochs_range, history["val_recall"], "^--", alpha=0.7, label="Recall", ms=3)
ax2.axvline(best_epoch, color="red", ls=":", label=f"best epoch={best_epoch}")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("Validation Metrics")
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Подбор порога

Порог 0.5 не обязательно оптимален. Подбираем на валидации.

In [ ]:
# Загружаем лучшие веса
model.load_state_dict(best_state)
model.to(device)
model.eval()

MIN_RECALL = 0.85
thresholds = np.arange(0.05, 0.96, 0.025)
val_results = []

for thr in thresholds:
    pred_fn = make_predict_fn(model, tokenizer, thr)
    res = evaluate(val_data, pred_fn)
    val_results.append(res)
    print(f"  threshold={thr:.3f}: P={res['Precision']:.3f} R={res['Recall']:.3f} F1={res['F1']:.3f}"
          f"  FP={res['FP']} FN={res['FN']}")

eligible = [(i, val_results[i]) for i in range(len(val_results))
            if val_results[i]["Recall"] >= MIN_RECALL]

if eligible:
    best_idx = max(eligible, key=lambda x: x[1]["F1"])[0]
else:
    best_idx = max(range(len(val_results)), key=lambda i: val_results[i]["F1"])
    print(f"\nНи один порог не даёт recall >= {MIN_RECALL}, берём лучший F1")

best_threshold = thresholds[best_idx]
print(f"\nОптимальный порог: {best_threshold:.3f}")
print(f"  P={val_results[best_idx]['Precision']:.3f} "
      f"R={val_results[best_idx]['Recall']:.3f} "
      f"F1={val_results[best_idx]['F1']:.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, [r["F1"] for r in val_results], "o-", label="F1", ms=4)
ax.plot(thresholds, [r["Precision"] for r in val_results], "s--", alpha=0.7, label="Precision", ms=3)
ax.plot(thresholds, [r["Recall"] for r in val_results], "^--", alpha=0.7, label="Recall", ms=3)
ax.axvline(best_threshold, color="red", ls=":", label=f"best={best_threshold:.3f}")
ax.axhline(MIN_RECALL, color="green", ls=":", alpha=0.5, label=f"min recall={MIN_RECALL}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Подбор порога на валидации")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Оценка на тестовой выборке

Переобучаем на полном train с лучшим числом эпох. Сравниваем со всеми предыдущими методами.

In [ ]:
print(f"Переобучение на полном train ({len(train_data)} текстов) за {best_epoch} эпох...")

# Освобождаем всю память от предыдущего обучения
for _var in ["model", "optimizer", "scheduler", "fit_loader", "val_loader"]:
    if _var in dir():
        del globals()[_var]
if device.type == "mps":
    torch.mps.empty_cache()

model_final = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=2)
model_final.gradient_checkpointing_enable()
model_final.to(device)

train_dataset = SentenceBoundaryDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

optimizer_final = AdamW(model_final.parameters(), lr=LR, weight_decay=0.01)
total_steps_final = len(train_loader) * best_epoch // GRAD_ACCUM
warmup_final = int(total_steps_final * WARMUP_RATIO)
scheduler_final = get_linear_schedule_with_warmup(optimizer_final, warmup_final, total_steps_final)

for epoch in range(best_epoch):
    model_final.train()
    total_loss = 0
    n_batches = 0
    optimizer_final.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_final(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits.view(-1, 2), labels.view(-1))
        loss = loss / GRAD_ACCUM
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model_final.parameters(), 1.0)
            optimizer_final.step()
            scheduler_final.step()
            optimizer_final.zero_grad()
            if device.type == "mps":
                torch.mps.empty_cache()

        total_loss += loss.item() * GRAD_ACCUM
        n_batches += 1

    print(f"  Epoch {epoch+1}/{best_epoch}: loss={total_loss/n_batches:.4f}")

# Оценка на тесте
model_final.eval()
predict_transformer = make_predict_fn(model_final, tokenizer, best_threshold)
test_result = evaluate(test_data, predict_transformer)

print(f"\n=== Transformer (rubert-base, threshold={best_threshold:.3f}) ===")
print(f"  TP={test_result['TP']}, FP={test_result['FP']}, FN={test_result['FN']}")
print(f"  Precision = {test_result['Precision']:.3f}")
print(f"  Recall    = {test_result['Recall']:.3f}")
print(f"  F1        = {test_result['F1']:.3f}")

In [ ]:
# Сравнение всех методов
EOS_SET = set(".!?\u2026")
DASHES_SET = set("\u2014\u2013\u2212-")

def baseline_standard(text):
    boundaries = set()
    for i in range(1, len(text) - 1):
        if text[i - 1] in EOS_SET and text[i] == " " and text[i + 1].isupper():
            boundaries.add(i + 1)
    return boundaries

def baseline_extended(text):
    boundaries = baseline_standard(text)
    for i in range(1, len(text) - 1):
        if text[i - 1] in DASHES_SET and text[i] == " " and text[i + 1].isupper():
            boundaries.add(i + 1)
        if text[i - 1] in DASHES_SET and text[i].isupper():
            boundaries.add(i)
        if text[i - 1] == "\u00ab" and text[i].isupper():
            if i >= 2 and (text[i - 2] == " " or text[i - 2] in EOS_SET):
                boundaries.add(i - 1)
        if text[i - 1] == "\u00bb" and text[i] == " " and text[i + 1].isupper():
            boundaries.add(i + 1)
    return boundaries

res_standard = evaluate(test_data, baseline_standard)
res_extended = evaluate(test_data, baseline_extended)

print(f"{'Метод':<35} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 68)
for name, res in [("standard (правило)", res_standard),
                  ("extended (правило)", res_extended),
                  ("CatBoost (notebook 04)", {"Precision": 0.977, "Recall": 0.990, "F1": 0.984}),
                  ("rubert-tiny2 (notebook 05)", {"Precision": 0.950, "Recall": 0.957, "F1": 0.953}),
                  ("rubert-base (this notebook)", test_result)]:
    print(f"{name:<35} {res['Precision']:>10.3f} {res['Recall']:>10.3f} {res['F1']:>10.3f}")

## 8. Анализ ошибок

In [ ]:
import random as rnd
rnd.seed(SEED)

fn_examples = []
fp_examples = []

for rec in test_data:
    ct = rec["clean_text"]
    true_bd = get_true_boundaries(rec)
    pred_bd = predict_transformer(ct)

    matched_true = set()
    matched_pred = set()
    for pb in pred_bd:
        for tb in true_bd:
            if abs(pb - tb) <= 2 and tb not in matched_true:
                matched_true.add(tb)
                matched_pred.add(pb)
                break

    for tb in true_bd - matched_true:
        context = ct[max(0, tb - 30):min(len(ct), tb + 30)]
        fn_examples.append(context)

    for pb in pred_bd - matched_pred:
        context = ct[max(0, pb - 30):min(len(ct), pb + 30)]
        fp_examples.append(context)

print(f"Пропущенных границ (FN): {len(fn_examples)}")
if fn_examples:
    print("\nПримеры FN:")
    for ex in rnd.sample(fn_examples, min(10, len(fn_examples))):
        print(f"  ...{ex}...")

print(f"\nЛожных срабатываний (FP): {len(fp_examples)}")
if fp_examples:
    print("\nПримеры FP:")
    for ex in rnd.sample(fp_examples, min(10, len(fp_examples))):
        print(f"  ...{ex}...")

## 9. Скорость инференса

In [ ]:
model_final.to("cpu")
model_final.eval()
predict_cpu = make_predict_fn(model_final, tokenizer, best_threshold)

# Прогрев
_ = predict_cpu(test_data[0]["clean_text"])

times = []
for rec in test_data:
    start = time.perf_counter()
    _ = predict_cpu(rec["clean_text"])
    elapsed = time.perf_counter() - start
    times.append(elapsed)

times = np.array(times)
print(f"Inference на CPU ({len(test_data)} текстов):")
print(f"  Mean:  {times.mean()*1000:.0f} ms")
print(f"  P50:   {np.percentile(times, 50)*1000:.0f} ms")
print(f"  P95:   {np.percentile(times, 95)*1000:.0f} ms")
print(f"  Max:   {times.max()*1000:.0f} ms")
print(f"\nВсе тексты < 2 сек: {(times < 2).all()}")

## 10. Итоги

| Метод | Precision | Recall | F1 |
|-------|-----------|--------|-----|
| standard (правило) | 0.977 | 0.894 | 0.933 |
| extended (правило) | 0.924 | 0.975 | 0.949 |
| CatBoost (notebook 04) | 0.977 | 0.990 | 0.984 |
| rubert-tiny2 (notebook 05) | 0.950 | 0.957 | 0.953 |
| **rubert-base (this notebook)** | **?** | **?** | **?** |

Таблица заполнится после запуска ноутбука.